In [ ]:
import glob
import pandas as pd
import os
import re
import json
import numpy as np

from IPython.display import display
from tqdm import tqdm

def extract_model_answer_from_api_response(body):    
    if "output" in body:
        msgs = [x for x in body["output"] if x["type"] == "message"]
        assert len(msgs) >= 1, body
        text = msgs[-1]["content"][0]["text"]
    elif "choices" in body:
        text = body["choices"][0]["message"]["content"]
        if text is None:
            return ""
    else:
        raise ValueError("Failed to extract answer from body\n\n" + str(body))
    return text


def get_token_usage(usage_dict):
    # extract token usage gpt-5 aware (different output format)
    if "prompt_tokens" in usage_dict:
        return usage_dict["prompt_tokens"], usage_dict["completion_tokens"]
    elif "input_tokens" in usage_dict:
        return usage_dict["input_tokens"], usage_dict["output_tokens"]
    else:
        raise ValueError("Unable to find token usage fields")


import statsmodels.api as sm
from itertools import combinations
from statsmodels.stats.multitest import multipletests
import numpy as np

def get_pval_matrix_mcnemar(df, target_column, index_cols, group_column, correction_method=None, alpha=0.05):
    '''
    Performs pairwise McNemar tests and optionally applies a multiple comparison correction.

    Args:
        df (pd.DataFrame): Results of all classifiers. One row = one sample.
        target_column (str): The column with binary outcomes (e.g., 'is_correct').
        index_cols (list): Columns that uniquely identify each sample across groups.
        group_column (str): The column to compare pairwise (e.g., 'prompt_type').
        correction_method (str, optional): The method for multiple comparison correction. 
                                           Examples: 'bonferroni', 'fdr_bh' (Benjamini-Hochberg). 
                                           If None, no correction is applied. Defaults to None.
        alpha (float): The significance level for determining rejection of the null hypothesis.

    Returns:
        tuple: 
            - pd.DataFrame: A matrix of p-values (corrected, if a method was specified).
            - pd.DataFrame or None: A summary DataFrame of the pairwise comparisons and their 
                                    significance, if a correction method was used. 
                                    Otherwise, None.
    '''
    df = df.copy().set_index(index_cols)
    g = df.groupby(group_column)
    classifiers = list(g.groups.keys())
    
    # Store raw p-values and their corresponding pairs
    raw_pvals_list = []
    pairs = []

    for left_class, right_class in combinations(classifiers, 2):
        pairs.append((left_class, right_class))
        left = g.get_group(left_class)
        right = g.get_group(right_class)
        
        aligned_predictions = left[[target_column]].merge(
            right[[target_column]], left_index=True, right_index=True, suffixes=("_l", "_r")
        )
        
        table = pd.crosstab(
            aligned_predictions[target_column + '_l'], 
            aligned_predictions[target_column + '_r']
        )
        
        try:
            # Note: correction=True applies Yates's continuity correction for the chi-squared test
            stats = sm.stats.mcnemar(table, exact=False, correction=True)
            raw_pvals_list.append(stats.pvalue)
        except ValueError:
            # This can happen if the contingency table is not 2x2.
            # A p-value of 1.0 indicates no significant difference.
            raw_pvals_list.append(1.0)

    # If no correction is needed, just build the raw p-value matrix and return
    if correction_method is None:
        pvals_matrix = pd.DataFrame(1.0, index=classifiers, columns=classifiers, dtype="float")
        for (p1, p2), p in zip(pairs, raw_pvals_list):
            pvals_matrix.loc[p1, p2] = p
            pvals_matrix.loc[p2, p1] = p
        return pvals_matrix, None

    # --- Apply Multiple Comparison Correction ---
    reject, pvals_corrected, _, _ = multipletests(
        raw_pvals_list, 
        alpha=alpha, 
        method=correction_method
    )

    # Create the corrected p-value matrix
    corrected_pvals_matrix = pd.DataFrame(1.0, index=classifiers, columns=classifiers, dtype="float")
    for i, (p1, p2) in enumerate(pairs):
        corrected_pvals_matrix.loc[p1, p2] = pvals_corrected[i]
        corrected_pvals_matrix.loc[p2, p1] = pvals_corrected[i]
        
    # Create the summary DataFrame
    summary_df = pd.DataFrame({
        'pair': [f"{p1} vs {p2}" for p1, p2 in pairs],
        'raw_p_value': raw_pvals_list,
        'corrected_p_value': pvals_corrected,
        f'significant_at_{alpha}': reject
    })
    
    return corrected_pvals_matrix, summary_df

import numpy as np
import pandas as pd
from itertools import combinations
from multiprocess import Pool, cpu_count
from statsmodels.stats.multitest import multipletests

def f1_fast(y_true, y_pred):
    tp = np.count_nonzero(y_true & y_pred)
    fp = np.count_nonzero(~y_true & y_pred)
    fn = np.count_nonzero(y_true & ~y_pred)
    denom = 2*tp + fp + fn
    return 0.0 if denom == 0 else (2.0*tp) / denom

def diff_f1_fast(y_true, p1, p2):
    return f1_fast(y_true, p1) - f1_fast(y_true, p2)

_G = None
def _init(g):  # g: dict[label] -> pd.DataFrame[['y_true','y_pred']] indexed
    global _G; _G = g

def _pval_task(task):
    l, r, seed, n_resamples = task
    a, b = _G[l], _G[r]
    idx = a.index.intersection(b.index)
    if len(idx) == 0:
        return (l, r, np.nan)
    y = a.loc[idx, "y_true"].to_numpy(np.bool_)
    p1 = a.loc[idx, "y_pred"].to_numpy(np.bool_)
    p2 = b.loc[idx, "y_pred"].to_numpy(np.bool_)
    n = y.shape[0]
    rng = np.random.default_rng(seed)
    # percentile bootstrap of diff; two-sided p-value around 0
    diffs = np.empty(n_resamples, dtype=float)
    for i in range(n_resamples):
        bidx = rng.integers(0, n, size=n)
        diffs[i] = diff_f1_fast(y[bidx], p1[bidx], p2[bidx])
    p = 2.0 * min((diffs >= 0).mean(), (diffs <= 0).mean())
    return (l, r, min(1.0, p))

def parallel_pvals_f1(
    df,
    index_cols,
    group_column,
    n_resamples=10000,
    n_workers=None,
    base_seed=12345,
    chunksize=8,
    alpha=0.05,
    correction="holm",  # options: "holm", "bonferroni", "fdr_bh", "fdr_by"
):
    """
    Parallel bootstrap p-values for all classifier pairs with multiple-testing correction.
    Returns:
      - sig: symmetric DataFrame[bool] of rejections after correction
      - pvals: Series (MultiIndex pairs) of raw p-values
      - pvals_adj: Series (MultiIndex pairs) of adjusted p-values
    """
    # Prepare per-group frames
    df2 = df.set_index(index_cols)
    groups = {k: v.loc[:, ["y_true", "y_pred"]] for k, v in df2.groupby(group_column)}

    classifiers = list(groups.keys())
    if len(classifiers) < 2:
        sig = pd.DataFrame(False, index=classifiers, columns=classifiers, dtype=bool)
        pvals = pd.Series(dtype=float)
        pvals_adj = pd.Series(dtype=float)
        return sig, pvals, pvals_adj

    pairs = list(combinations(classifiers, 2))
    tasks = [(l, r, base_seed + i, n_resamples) for i, (l, r) in enumerate(pairs)]

    pval_idx = pd.MultiIndex.from_tuples(pairs, names=["left", "right"])
    pvals = pd.Series(np.nan, index=pval_idx, dtype=float)

    if n_workers is None:
        n_workers = max(1, cpu_count())

    with Pool(processes=n_workers, initializer=_init, initargs=(groups,)) as pool:
        for l, r, p in pool.imap_unordered(_pval_task, tasks, chunksize=chunksize):
            pvals[(l, r)] = p

    # Multiple-testing correction (ignore NaNs from non-overlapping pairs)
    mask = pvals.notna()
    pvals_adj = pd.Series(np.nan, index=pvals.index, dtype=float)
    rejections = pd.Series(False, index=pvals.index, dtype=bool)

    if mask.any():
        reject, pval_corr, _, _ = multipletests(pvals[mask].values, alpha=alpha, method=correction)
        pvals_adj[mask] = pval_corr
        rejections[mask] = reject

    # Build symmetric significance matrix
    sig = pd.DataFrame(False, index=classifiers, columns=classifiers, dtype=bool)
    for (l, r), rej in rejections.items():
        sig.loc[l, r] = sig.loc[r, l] = bool(rej)

    # Zero diagonal
    for c in classifiers:
        sig.loc[c, c] = False

    return sig, pvals, pvals_adj


import networkx as nx
import pulp

def insignificant_cliques(significance_df):

    groups = significance_df.apply(lambda x: frozenset(x[x==""].index.difference([x.name])))
    groups = groups[groups.apply(len) > 0]
    df = groups.explode().to_frame(name="right").reset_index(names="left")

    G = nx.Graph()
    G.add_edges_from(df[['left','right']].itertuples(index=False, name=None))
    cliques = list(nx.find_cliques(G))
    edges = list(G.edges())

    model = pulp.LpProblem('ECC', pulp.LpMinimize)
    x = pulp.LpVariable.dicts('x', range(len(cliques)), cat='Binary')

    model += pulp.lpSum(x[i] for i in x)
    for u, v in edges:
        model += pulp.lpSum(x[i] for i, c in enumerate(cliques) if u in c and v in c) >= 1

    model.solve(pulp.PULP_CBC_CMD(msg=False))
    return [cliques[i] for i in x if pulp.value(x[i]) > 0.5]


# name: is_reasoning_model
MODELS = {
 'gpt-4.1-2025-04-14': False,
 'gpt-4.1-mini-2025-04-14': False,
 'gpt-4.1-nano-2025-04-14': False,
 'gpt-5-2025-08-07': True,
 'gpt-5-mini-2025-08-07': True,
 'gpt-5-nano-2025-08-07': True,
 'o3-mini-2025-01-31': True,
 'o4-mini-2025-04-16': True,
 'gpt-4o-2024-08-06': False,
 'Qwen/Qwen3-235B-A22B-Instruct-2507-FP8': False,
 'Qwen/Qwen3-30B-A3B-Instruct-2507-FP8': False,
 'Qwen/Qwen3-30B-A3B-Thinking-2507-FP8': True,
 'Qwen/Qwen3-4B-Thinking-2507-FP8': True,
 'Qwen/Qwen3-Next-80B-A3B-Instruct-FP8': False,
 'Qwen/Qwen3-Next-80B-A3B-Thinking-FP8': True,
 'RedHatAI/Llama-3.3-70B-Instruct-FP8-dynamic': False,
 'meta-llama/Llama-3.1-8B-Instruct': False,
 'microsoft/phi-4': False,
 'mistralai/Mistral-Small-24B-Instruct-2501': False,
 'openai/gpt-oss-120b': True,
 'openai/gpt-oss-20b': True,
 'zai-org/GLM-4-32B-0414': False,
 'zai-org/GLM-Z1-32B-0414': True,
}

# Pharma
***

In [ ]:
# for pretty plots and tables

pharma_prompt_map = dict(P_E1="RAG",P_E2="RAG+",P_E3="5 Examples",P_E4="DELETE",P_E5="10 Examples", 
                         P_M="Mechanism",P_X="Direct", P_H="Hybrid", P_HAI="Hybrid by AI", P_MAI="AI Mechanism")

## Load ground truth
***

In [ ]:
# load ground truth files
DRUG_ID_MAP = {
    "upadacitinib": "DB15091",
    "digitoxin": "DB01396",
    "simvastatin": "DB00641",
}

def read_pharma_gt_csv(x):
    gt = pd.read_csv(x)
    dataset = x.split("/")[1]
    gt["DRUG_A"] = dataset
    gt["DRUG_A_ID"] = DRUG_ID_MAP[dataset]
    gt["GT"] = gt.GT.apply(lambda x: x.lower())
    return gt

gt_csvs = glob.glob("phase1/**/*final_dataset.csv", recursive=False)


pharma_gt = pd.concat([read_pharma_gt_csv(x) for x in gt_csvs])
pharma_gt.head(2)

## Load model responses
***

In [ ]:
PATTERN = re.compile(r"\?\?\s*(yes|no)\s*\?\?")
def extract_canonicalized_pharma_answer(text):
    text = text.lower()
    matches = list(PATTERN.finditer(text))
    if not matches:
        return "unk"
    return matches[-1].group(1)


In [ ]:
def read_pharma_model_jsonl(x):
    with open(x, "r") as f:
        lines = f.readlines()
    lines = [json.loads(x) for x in lines]
    data = pd.DataFrame(lines)
    data = data.drop("id", axis=1)
    
    data["PROMPT"] = data.custom_id.apply(lambda x: "P_" + x.split("P_")[1].split("_")[0])
    data["MODEL"] = data.response.apply(lambda x: x["body"]["model"])
    data["dataset"] = x.split("/")[1]    
    
    data["text_answer"] = data.response.apply(lambda x: extract_model_answer_from_api_response(x["body"]))
    data["extracted_answer"] = data.text_answer.apply(extract_canonicalized_pharma_answer)
    data[["prompt_tokens", "completion_tokens"]] = data.response.apply(lambda x: get_token_usage(x["body"]["usage"])).apply(pd.Series)
    data[["DRUG_A_ID", "DRUG_B_ID"]] = data.custom_id.apply(lambda x: pd.Series(x.split("__")[:2]))
    
    return data

csvs = [x for x in glob.glob("merged/**/batches_output.jsonl", recursive=True) if x.split("/")[1] in DRUG_ID_MAP]

pharma_prompts_to_show = ["P_E1","P_E2", "P_E5","P_M","P_E3","P_MAI","P_X","P_H"]


In [ ]:
# load all model responses
pharma_responses = pd.concat([read_pharma_model_jsonl(x) for x in tqdm(csvs)])

# add ground truth labels
pharma_responses = pharma_responses.merge(pharma_gt, on=["DRUG_A_ID","DRUG_B_ID"])


In [ ]:
num_datasets_per_model = pharma_responses.groupby("MODEL").dataset.agg(lambda s: len(s.unique()))
if (num_datasets_per_model < 3).any():
    print("Filtering out models that didn't answer all datasets:")
    print(num_datasets_per_model[num_datasets_per_model < 3].index.tolist())
    good_models = set(num_datasets_per_model[num_datasets_per_model == 3].index)
    pharma_responses = pharma_responses[pharma_responses.MODEL.isin(good_models)]
pharma_responses["is_reasoning"] = pharma_responses.MODEL.map(lambda x: MODELS.get(x))

In [ ]:
# remove non extractable responses
pr_clean = pharma_responses[pharma_responses.extracted_answer != "unk"].copy()

pr_clean["is_correct"] = pr_clean.extracted_answer == pr_clean.GT

pr_dirty = pharma_responses[pharma_responses.extracted_answer == "unk"].copy()

## Pharma main metrics
***

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
metrics = {
    "precision": precision_score,
    "recall": recall_score,
    "f1": f1_score
}

def compute_metrics(df):
    # Map yes/no → 1/0
    y_true = df["GT"].map({"yes": 1, "no": 0})
    y_pred = df["extracted_answer"].map({"yes": 1, "no": 0})
    d = {name: func(y_pred, y_true) for name, func in metrics.items()}
    d["completion_tokens"] = df.completion_tokens.mean()
    
    return pd.Series(d)

agg_cols = ["dataset", "MODEL", "PROMPT"]
g = pr_clean.groupby(agg_cols)
pharma_results = g.apply(compute_metrics, include_groups=False)
pharma_results["is_reasoning"] = pharma_results.index.map(lambda x: MODELS.get(x[1]))
pharma_results.head(2)

In [ ]:
def show_pharma_table(data):
    args = ["mean","std","count"]
    mean_per_prompt_and_model = (data.groupby(["PROMPT","MODEL"]).agg({"precision":args, "recall":args, "f1":args, "completion_tokens":args}) * 100)
    mean_per_prompt_and_model["completion_tokens"] /= 100
    mean_per_prompt_and_model.sort_values(("f1","mean"), ascending=False, inplace=True)

    mean_per_prompt = mean_per_prompt_and_model.groupby("PROMPT").agg("mean").sort_values(("f1","mean"), ascending=False)

    temp = mean_per_prompt.loc[pharma_prompts_to_show].sort_values(("f1","mean"), ascending=False)

    g = temp.rename(index=pharma_prompt_map).T.groupby(level=0)
    results_latex = g.apply(lambda df: df.T[df.T.columns.droplevel(1).unique().item()].apply(lambda row: fr"${row['mean']:0.2f}_{{\pm {row['std']:0.1f}}}$", axis=1)).T
    results_latex = results_latex.loc[[c for c in results_latex.index if c!="DELETE"]]
    results_latex.reset_index(inplace=True)

    results_latex = results_latex[["PROMPT","precision","recall","f1","completion_tokens"]]
    results_latex.rename({"f1":r"$\text{F1}_{\uparrow}$", "recall":r"$\text{Recall}_{\uparrow}$", "precision":r"$\text{Precision}_{\uparrow}$",
                          "completion_tokens":r"\#Output tokens", 
                          "PROMPT":"Prompt type"}, axis=1, inplace=True)

    display(results_latex)

    print(results_latex.to_latex(escape=False, index=False, column_format="lcccr"))
    return mean_per_prompt_and_model, mean_per_prompt

In [ ]:
print("-" * 30,"reasoning", "-" * 30)
mppm_reasoning, mpp_reasoning = show_pharma_table(pharma_results[pharma_results.is_reasoning])

print("-" * 30,"instruct", "-" * 30)
mppm_instruct, mpp_instruct = show_pharma_table(pharma_results[~pharma_results.is_reasoning])

### Significance Tests
***

In [ ]:
data = pr_clean[pr_clean.PROMPT.isin(pharma_prompts_to_show)].copy()

data["y_true"] = data.GT == "yes"
data["y_pred"] = data.extracted_answer == "yes"

In [ ]:
index_cols = ["MODEL","dataset","DRUG_A_ID","DRUG_B_ID"]
group_column = "PROMPT"
alpha = 0.05
correction_method="fdr_bh"

In [ ]:
significance, pvals, pvals_adj = parallel_pvals_f1(data[data.is_reasoning], index_cols, group_column, 10000, n_workers=16, 
                                                   chunksize=1, correction=correction_method, alpha=alpha)
print("-"*20,"REASONING","-"*20)
significance = significance.replace(True, "Yes").replace(False,"")
significance = significance.rename(pharma_prompt_map, axis=1).rename(index=pharma_prompt_map)
display(significance)
groups = insignificant_cliques(significance)
print("Non significant pairs:", [tuple(x) for x in groups])

In [ ]:
significance, pvals, pvals_adj = parallel_pvals_f1(data[~data.is_reasoning], index_cols, group_column, 10000, n_workers=16, 
                                                   chunksize=1, correction=correction_method, alpha=alpha)
print("-"*20,"INSTRUCT","-"*20)
significance = significance.replace(True, "Yes").replace(False,"")
significance = significance.rename(pharma_prompt_map, axis=1).rename(index=pharma_prompt_map)
display(significance)
groups = insignificant_cliques(significance)
print("Non significant pairs:", [tuple(x) for x in groups])

## Pharma extraction error stats
***

In [ ]:
pr_dirty["finish_reason"] = pr_dirty.response.apply(lambda x: x["body"]["choices"][0]["finish_reason"])
pr_dirty.groupby(["MODEL"]).finish_reason.value_counts().to_frame().groupby("finish_reason").sum() / len(pharma_responses) * 100

In [ ]:

errors = pr_dirty[pr_dirty.PROMPT.isin(pharma_prompts_to_show)]

errors_per_prompt = 100 - (errors.groupby(["PROMPT"]).custom_id.count().to_frame() / ncbi_responses.groupby(["PROMPT"]).custom_id.count().to_frame()).sort_values("custom_id",ascending=False).fillna(0) * 100
errors_per_prompt = errors_per_prompt[errors_per_prompt.index.isin(pharma_prompts_to_show)]
print(errors_per_prompt.round(2).rename(index=pharma_prompt_map).reset_index().to_latex(index=False, float_format=f"%.2f"))

In [ ]:
data = pr_clean[pr_clean.PROMPT.isin(pharma_prompts_to_show)].copy()
len(errors) / len(data) * 100

In [ ]:
pharma_error_stats = errors.groupby("MODEL").PROMPT.count().to_frame("#Extraction errors")
pharma_error_stats.sort_values("#Extraction errors", ascending=False).cumsum() / len(errors)

# NCBI Abstracts
***

In [ ]:
ncbi_prompt_map = dict(P_E3="5 Examples",P_E5="10 Examples", P_M="Mechanism",P_X="Direct", P_H="Hybrid", P_MAI2="Mechanism by AI")

In [ ]:
# read ground truth

pattern = re.compile(r"<([^>]+)>(.*?)<\/")

def extract_entities(text):
    keywords = ("SpecificDisease", "DiseaseClass", "CompositeMention", "Modifier")
    keyword_map = {k.lower(): k for k in keywords}
    
    
    matches = pattern.findall(text)
    r = []
    for bracket, span in matches:
        for kwl, kw in keyword_map.items():
            if kwl in bracket.lower():
                if len(span) == 0:
                    continue
                if span[0] == "[" and span[-1] == "]":
                    span = span[1:-1]
                r.append(dict(type=kw, text=span))
    return r

CAT = re.compile(r'<category="([^"]+)">(.*?)</category>', flags=re.DOTALL)

def strip_category_tags(text: str):
    # replace the whole tag with just its inner text
    return CAT.sub(r'\2', text or "")
    
def read_ncbi_corpus(fn):
    records = []
    with open(fn, encoding="utf-8") as fh:
        for line in fh:
            pmid, title, abstract = line.rstrip("\n").split("\t", 2)
            records.append({
                "pmid": pmid,
                "title": title,
                "abstract": abstract,
            })

    df = pd.DataFrame(records)

    for col in ("title", "abstract"):
        df[f"{col}_clean"]    = df[col].apply(strip_category_tags)
        df[f"{col}_entities"] = df[col].apply(extract_entities)
    return df

In [ ]:
def extract_last_abstract(text):
    end = text.rfind("</abstract>")
    start = text.rfind("<abstract>")
    if end < 0 or start < 0 or end < start:
        return "unk"
    r = text[start+10:end].strip()
    return r

In [ ]:
def read_ncbi_model_jsonl(x):
    with open(x, "r") as f:
        lines = f.readlines()
    lines = [json.loads(x) for x in lines]
    data = pd.DataFrame(lines)
    data = data.drop("id", axis=1)

    data["MODEL"] = data.response.apply(lambda x: x["body"]["model"])
    data["dataset"] = x.split("/")[1]    

    data["text_answer"] = data.response.apply(lambda x: extract_model_answer_from_api_response(x["body"]))
    data["extracted_answer"] = data.text_answer.apply(extract_last_abstract)
    data["predicted_entities"] = data.extracted_answer.apply(extract_entities)

    data[["prompt_tokens", "completion_tokens"]] = data.response.apply(lambda x: get_token_usage(x["body"]["usage"])).apply(pd.Series)
    data[["pmid", "PROMPT"]] = data.custom_id.apply(lambda x: pd.Series(x.split("_",1)))

    return data

csvs = [x for x in glob.glob("merged/ncbi/**/batches_output.jsonl", recursive=True)]

In [ ]:
ncbi_responses = pd.concat([read_ncbi_model_jsonl(x) for x in tqdm(csvs)])

# add ground truth
ncbi_test = read_ncbi_corpus("ncbi/NCBI_corpus_testing.txt")
ncbi_responses = ncbi_responses.merge(ncbi_test[["pmid","abstract","abstract_clean","abstract_entities"]], on="pmid")

ncbi_responses["is_reasoning"] = ncbi_responses.MODEL.map(lambda x: MODELS.get(x))

In [ ]:
from collections import Counter

def precision_recall_f1_em(y_true, y_pred, soft=False):
    # metrics based on exact match
    # y_true, y_pred are lists of extracted entity mentions (dicts with type and text field)
    if soft:
        y_true = Counter([(x["text"].lower()) for x in y_true])
        y_pred = Counter([(x["text"].lower()) for x in y_pred])
    else:
        y_true = Counter([(x["type"].lower().strip(), x["text"].lower().strip()) for x in y_true])
        y_pred = Counter([(x["type"].lower().strip(), x["text"].lower().strip()) for x in y_pred])
    true_positives = sum((y_true & y_pred).values())
    total_predicted = sum(y_pred.values())
    total_actual = sum(y_true.values())

    precision = true_positives / total_predicted if total_predicted > 0 else 0
    recall = true_positives / total_actual if total_actual > 0 else 0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1_score

In [ ]:
ncbi_clean = ncbi_responses[ncbi_responses.extracted_answer != "unk"].copy()
ncbi_dirty = ncbi_responses[ncbi_responses.extracted_answer == "unk"].copy()
# Error rate
print(len(ncbi_dirty) / len(ncbi_responses) * 100)

## NCBI Metrics
***

In [ ]:
ncbi_clean[["precision","recall","f1"]] = (
    ncbi_clean.apply(lambda row: pd.Series(precision_recall_f1_em(row.abstract_entities, row.predicted_entities, soft=False)), axis=1)
)
ncbi_prompts_to_show = ["P_E5","P_M","P_E3","P_MAI2","P_X","P_H"]

In [ ]:
def show_ncbi_table(data):
    # reasoning
    args = ["mean","std","count"]
    mean_per_prompt_and_model = (data.groupby(["PROMPT","MODEL"]).agg({"precision":args, "recall":args, "f1":args, "completion_tokens":args}) * 100)
    mean_per_prompt_and_model["completion_tokens"] /= 100
    mean_per_prompt_and_model.sort_values(("f1","mean"), ascending=False, inplace=True)
    
    mean_per_prompt = mean_per_prompt_and_model.groupby("PROMPT").agg("mean").sort_values(("f1","mean"), ascending=False)
    
    temp = mean_per_prompt.loc[ncbi_prompts_to_show].sort_values(("f1","mean"), ascending=False)
    g = temp.rename(index=ncbi_prompt_map).T.groupby(level=0)
    results_latex = g.apply(lambda df: df.T[df.T.columns.droplevel(1).unique().item()].apply(lambda row: fr"${row['mean']:0.2f}_{{\pm {row['std']:0.1f}}}$", axis=1)).T
    results_latex.reset_index(inplace=True)
    
    results_latex = results_latex[["PROMPT","precision","recall","f1","completion_tokens"]]
    results_latex.rename({"f1":r"$\text{F1}_{\uparrow}$", "recall":r"$\text{Recall}_{\uparrow}$", "precision":r"$\text{Precision}_{\uparrow}$",
                          "completion_tokens":r"\#Output tokens", 
                          "PROMPT":"Prompt type"}, axis=1, inplace=True)
    
    display(results_latex)
    
    print(results_latex.to_latex(escape=False, index=False, column_format="lcccr"))

In [ ]:
print("-" * 30,"reasoning", "-" * 30)
show_ncbi_table(ncbi_clean[ncbi_clean.is_reasoning])

print("-" * 30,"instruct", "-" * 30)
show_ncbi_table(ncbi_clean[~ncbi_clean.is_reasoning])

### Significance Tests
***

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from multiprocess import Pool, cpu_count
from statsmodels.stats.multitest import multipletests

# Worker globals + init
_G = None
def _init(groups):
    # groups: dict[label] -> pd.Series or DataFrame[['f1']] indexed by index_cols
    global _G
    _G = groups

def _pval_task(task):
    l, r, seed, n_resamples, chunk = task
    a, b = _G[l], _G[r]
    idx = a.index.intersection(b.index)
    if len(idx) < 2:
        # Not enough paired units to bootstrap reliably
        return (l, r, np.nan, np.nan)

    s1 = a.loc[idx].to_numpy(dtype=float).ravel()
    s2 = b.loc[idx].to_numpy(dtype=float).ravel()
    n = s1.shape[0]

    # Observed paired mean difference
    d0 = s1.mean() - s2.mean()

    rng = np.random.default_rng(seed)

    # Vectorized bootstrap in memory-bounded chunks
    diffs = np.empty(n_resamples, dtype=float)
    out_off = 0
    while out_off < n_resamples:
        m = min(chunk, n_resamples - out_off)
        bidx = rng.integers(0, n, size=(m, n))
        diffs[out_off:out_off + m] = s1[bidx].mean(axis=1) - s2[bidx].mean(axis=1)
        out_off += m

    # Two-sided percentile-style p-value for H0: mean diff = 0
    # (common simple bootstrap p-value; conservative but practical)
    p = 2.0 * min((diffs >= 0).mean(), (diffs <= 0).mean())
    p = float(min(1.0, p))

    return (l, r, p, d0)

def parallel_pvals_f1(
    df,
    index_cols,
    group_column,
    f1_column="f1",
    n_resamples=10000,
    n_workers=None,
    base_seed=12345,
    chunksize=8,
    alpha=0.05,
    correction="holm",  # "holm", "bonferroni", "fdr_bh", "fdr_by"
    boot_chunk=2048,    # controls bootstrap vectorization memory
):
    """
    Parallel bootstrap p-values for all classifier pairs using paired mean F1 differences.
    Multiple-testing correction is applied inside and results returned.

    Inputs:
      - df: rows contain F1 scores per (index_cols, group_column)
      - index_cols: columns identifying the paired unit (e.g., fold, seed)
      - group_column: classifier label column
      - f1_column: column with the F1 score (float)

    Returns:
      - sig: symmetric DataFrame[bool] of rejections after correction
      - pvals: Series (MultiIndex pairs) of raw p-values
      - pvals_adj: Series (MultiIndex pairs) of adjusted p-values
      - diffs: Series (MultiIndex pairs) of observed mean F1 differences (left - right)
    """
    # Prepare per-group Series of F1, indexed by index_cols
    df2 = df.set_index(index_cols)
    groups = {k: v[f1_column].astype(float) for k, v in df2.groupby(group_column)}

    classifiers = list(groups.keys())
    if len(classifiers) < 2:
        sig = pd.DataFrame(False, index=classifiers, columns=classifiers, dtype=bool)
        pvals = pd.Series(dtype=float)
        pvals_adj = pd.Series(dtype=float)
        diffs = pd.Series(dtype=float)
        return sig, pvals, pvals_adj, diffs

    pairs = list(combinations(classifiers, 2))
    tasks = [(l, r, base_seed + i, n_resamples, boot_chunk) for i, (l, r) in enumerate(pairs)]

    pval_idx = pd.MultiIndex.from_tuples(pairs, names=["left", "right"])
    pvals = pd.Series(np.nan, index=pval_idx, dtype=float)
    diffs = pd.Series(np.nan, index=pval_idx, dtype=float)

    if n_workers is None:
        n_workers = max(1, cpu_count())

    with Pool(processes=n_workers, initializer=_init, initargs=(groups,)) as pool:
        for l, r, p, d0 in pool.imap_unordered(_pval_task, tasks, chunksize=chunksize):
            pvals[(l, r)] = p
            diffs[(l, r)] = d0

    # Multiple-testing correction (ignore NaNs from non-overlapping/too-few pairs)
    mask = pvals.notna()
    pvals_adj = pd.Series(np.nan, index=pvals.index, dtype=float)
    rejections = pd.Series(False, index=pvals.index, dtype=bool)

    if mask.any():
        reject, pval_corr, _, _ = multipletests(
            pvals[mask].values, alpha=alpha, method=correction
        )
        pvals_adj[mask] = pval_corr
        rejections[mask] = reject

    # Build symmetric significance matrix
    sig = pd.DataFrame(False, index=classifiers, columns=classifiers, dtype=bool)
    for (l, r), rej in rejections.items():
        sig.loc[l, r] = sig.loc[r, l] = bool(rej)
    for c in classifiers:
        sig.loc[c, c] = False

    return sig, pvals, pvals_adj, diffs

In [ ]:
data = ncbi_clean[ncbi_clean.PROMPT.isin(ncbi_prompts_to_show)]

In [ ]:
index_cols = ["MODEL","dataset","pmid"]
group_column = "PROMPT"
alpha = 0.05
correction_method="fdr_bh"

In [ ]:
significance, pvals, pvals_adj,diffs = parallel_pvals_f1(data[data.is_reasoning], index_cols, group_column, "f1", 10000, 
                                                         n_workers=16, chunksize=1, correction=correction_method, alpha=alpha, )
print("-"*20,"REASONING","-"*20)
significance = significance.replace(True, "Yes").replace(False,"")
significance = significance.rename(ncbi_prompt_map, axis=1).rename(index=ncbi_prompt_map)
display(significance)
groups = insignificant_cliques(significance)
print("Non significant pairs:", [tuple(x) for x in groups])

In [ ]:
significance, pvals, pvals_adj,diffs = parallel_pvals_f1(data[~data.is_reasoning], index_cols, group_column, "f1", 10000, 
                                                         n_workers=16, chunksize=1, correction=correction_method, alpha=alpha, )
print("-"*20,"INSTRUCT","-"*20)
significance = significance.replace(True, "Yes").replace(False,"")
significance = significance.rename(ncbi_prompt_map, axis=1).rename(index=ncbi_prompt_map)
display(significance)
groups = insignificant_cliques(significance)
print("Non significant pairs:", [tuple(x) for x in groups])

## NCBI Extraction Error Stats
***

In [ ]:
ncbi_dirty["finish_reason"] = ncbi_dirty.response.apply(lambda x: x["body"]["choices"][0]["finish_reason"])

errors = ncbi_dirty[ncbi_dirty.PROMPT.isin(ncbi_prompts_to_show)]

errors_per_prompt = 100 - (errors.groupby(["PROMPT"]).pmid.count().to_frame() / ncbi_responses.groupby(["PROMPT"]).pmid.count().to_frame()).sort_values("pmid",ascending=False).fillna(0) * 100
errors_per_prompt = errors_per_prompt[errors_per_prompt.index.isin(ncbi_prompt_map)]

print(errors_per_prompt.round(2).rename(index=ncbi_prompt_map).reset_index().to_latex(index=False, float_format=f"%.2f"))

# Math
***

In [ ]:
INT_MARK_RE = re.compile(r"\?\?\s*(-?\d[\d,]*)\s*\?\?")

def extract_pred_int(text: str) -> int | None:
    m = INT_MARK_RE.findall(text or "")
    if not m:
        return np.nan
    try:
        return int(m[-1].replace(",", ""))
    except Exception:
        return np.nan

CID_RE = re.compile(r"^(?P<kind>P_[A-Za-z0-9]+)__(?P<a>\d+)__(?P<b>\d+)__(?P<idx>\d+)$")

def parse_cid(cid: str):
    m = CID_RE.match(cid or "")
    if not m:
        return None
    return m.group("kind"), int(m.group("a")), int(m.group("b")), int(m.group("idx"))

In [ ]:
def read_math_model_jsonl(x):
    with open(x, "r") as f:
        lines = f.readlines()
    lines = [json.loads(x) for x in lines]
    data = pd.DataFrame(lines)
    data = data.drop("id", axis=1)
    
    data["MODEL"] = data.response.apply(lambda x: x["body"]["model"])
    data["dataset"] = x.split("/")[1]    
    
    data["text_answer"] = data.response.apply(lambda x: extract_model_answer_from_api_response(x["body"]))
    data[["PROMPT","a","b","idx"]] = data.custom_id.apply(lambda x: pd.Series(parse_cid(x)))
    data["GT"] = data["a"] * data["b"]
    data["y_pred"] = data.text_answer.apply(extract_pred_int)
    data[["prompt_tokens", "completion_tokens"]] = data.response.apply(lambda x: get_token_usage(x["body"]["usage"])).apply(pd.Series)

    return data

math_prompt_map = dict(P_E5="5 Examples",P_E5w="5 wrong examples",P_E10="10 Examples", P_M="Mechanism",P_X="Direct",
                       P_H2="Hybrid 2x2", P_H5="Hybrid 5x5", P_H7="Hybrid 7x7")

In [ ]:
csvs = [x for x in glob.glob("merged/math7/**/batches_output.jsonl", recursive=True)]

math_data = pd.concat([read_math_model_jsonl(x) for x in tqdm(csvs)])

# remove prompt type P_E5w, since it was just for fun...
math_data = math_data[math_data.PROMPT!="P_E5w"]

math_data["is_reasoning"] = math_data.MODEL.map(lambda x: MODELS.get(x))

In [ ]:
math_clean = math_data[math_data.y_pred.notna()].copy()
math_dirty = math_data[math_data.y_pred.isna()]

math_clean["is_correct"] = (math_clean["GT"] == math_clean["y_pred"])

## Math metrics
***

In [ ]:
def show_math_table(data):
    args = ["mean","std","count"]
    mean_per_prompt_and_model = data.groupby(["PROMPT","MODEL"])[["is_correct","completion_tokens"]].agg(args) * 100
    mean_per_prompt_and_model["completion_tokens"] /= 100
    mean_per_prompt_and_model.sort_values(("is_correct","mean"), ascending=False, inplace=True)
    
    mean_per_prompt = mean_per_prompt_and_model.groupby("PROMPT").agg("mean").sort_values(("is_correct","mean"), ascending=False)

    g = mean_per_prompt.rename(index=math_prompt_map).T.groupby(level=0)
    results_latex = g.apply(lambda df: df.T[df.T.columns.droplevel(1).unique().item()].apply(lambda row: fr"${row['mean']:0.2f}_{{\pm {row['std']:0.1f}}}$", axis=1)).T
    results_latex = results_latex.reindex(columns=results_latex.columns[::-1])
    results_latex.reset_index(inplace=True)
    
    results_latex.rename({"is_correct":r"$\text{Accuracy}_{\uparrow}$", 
                          "completion_tokens":r"\#Output tokens", 
                          "PROMPT":"Prompt type"}, axis=1, inplace=True)
    
    display(results_latex)
    
    print(results_latex.to_latex(escape=False, index=False, column_format="lcr"))

In [ ]:
print("-" * 30,"reasoning", "-" * 30)
show_math_table(math_clean[math_clean.is_reasoning])

print("-" * 30,"instruct", "-" * 30)
show_math_table(math_clean[~math_clean.is_reasoning])

### Significance tests
***
Hypothesis to test: Are outcomes of different prompt types actually different?
* --> Mcnemar test on aligned sequences

In [ ]:
alpha = 0.05
correction_method="fdr_bh"

pvals,summary = get_pval_matrix_mcnemar(math_clean[math_clean.is_reasoning], "is_correct", ["MODEL","a","b"], "PROMPT", 
                                        correction_method=correction_method, alpha=alpha)
significance = (pvals < alpha).replace(True, "Yes").replace(False,"")
print("-"*20,"REASONING","-"*20)
significance = significance.rename(math_prompt_map, axis=1).rename(index=math_prompt_map)
display(significance)
groups = insignificant_cliques(significance)
print("Non significant pairs:", [tuple(x) for x in groups])

pvals,summary = get_pval_matrix_mcnemar(math_clean[~math_clean.is_reasoning], "is_correct", ["MODEL","a","b"], "PROMPT", 
                                        correction_method=correction_method, alpha=alpha)
significance = (pvals < alpha).replace(True, "Yes").replace(False,"")
print("-"*20,"NON REASONING","-"*20)
significance = significance.rename(math_prompt_map, axis=1).rename(index=math_prompt_map)
display(significance)
groups = insignificant_cliques(significance)
print("Non significant pairs:", [tuple(x) for x in groups])

## Math extraction error stats
***

In [ ]:
math_error_rates = (math_dirty.groupby("MODEL")["a"].count() / math_data.groupby("MODEL").a.count()).fillna(0)
(math_error_rates.sort_values() * 100).round(2)